# Notebook 2 : Pandas avancé

In [1]:
import pandas as pd

## Tennis

Nous considérons les données des résultats des matchs de tennis masculin des tournois de Roland Garros et Wimbledon en 2013. La liste des variables et leur signification se trouvent sur [cette page](https://archive.ics.uci.edu/dataset/300/tennis+major+tournament+match+statistics) dans la section *Additional Variable Information*.

1. Commencer par charger le jeu de données relatif au tournoi de Roland Garros dans un dataframe `rg` à partir du fichier `rolandgarros2013.csv`.

In [2]:
rg=pd.read_csv('data/rolandgarros2013.csv',sep=',')

rg.head(10)

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,BPC.2,BPW.2,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2
0,Pablo Carreno-Busta,Roger Federer,1,0,0,3,62,27,38,11,...,7,7,14,18,88,6,6,6.0,NaN,NaN
1,Somdev Devvarman,Daniel Munoz-De La Nava,1,1,3,0,62,54,38,22,...,1,16,22,25,106,3,3,5.0,NaN,NaN
2,Tobias Kamke,Paolo Lorenzi,1,1,3,2,62,53,38,15,...,10,18,19,27,139,3,3,6.0,6.0,3.0
3,Julien Benneteau,Ricardas Berankis,1,1,3,1,72,87,28,19,...,4,13,33,43,149,6,3,7.0,6.0,NaN
4,Lukas Lacko,Sam Querrey,1,0,0,3,52,31,48,22,...,4,7,12,13,93,6,6,6.0,NaN,NaN
5,Jan Hajek,Denis Kudla,1,1,3,1,70,58,30,18,...,1,7,6,9,93,2,7,0.0,4.0,NaN
6,Adrian Mannarino,Pablo Cuevas,1,0,2,3,63,71,37,38,...,6,20,14,22,175,6,2,6.0,5.0,7.0
7,Gilles Simon,Lleyton Hewitt,1,1,3,2,59,42,41,25,...,9,10,19,35,120,6,6,4.0,1.0,5.0
8,Philipp Petzschner,Marin Cilic,1,0,0,3,56,27,44,13,...,7,12,13,21,92,6,6,6.0,NaN,NaN
9,Radek Stepanek,Nick Kyrgios,1,0,0,3,63,62,37,29,...,1,3,9,20,130,7,7,7.0,NaN,NaN


2. Afficher les noms des demi-finalistes.

In [3]:
rg.groupby('Round').agg(nb_lignes=pd.NamedAgg(column="Player1", aggfunc="count")).sort_values(by=['nb_lignes'],ascending=False)

,nb_lignes
Round,
1,63
2,31
3,16
4,8
5,4
6,2
7,1


In [4]:
rg[rg['Round']==6]

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,BPC.2,BPW.2,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2
122,David Ferrer,Jo-Wilfried Tsonga,6,1,3,0,60,35,40,23,...,2,5,7,16,84,1,6,2.0,NaN,NaN
123,Novak Djokovic,Rafael Nadal,6,0,2,3,67,76,33,30,...,8,16,15,26,177,6,3,6.0,6.0,9.0


3. Calculer le nombre moyen d'aces par match dans le tournoi.

In [5]:
rg['Ace_tot']=rg['ACE.1']+rg['ACE.2']
print(rg['Ace_tot'].mean())

12.688


4. Combien y a-t-il eu d'aces par match en moyenne à chaque niveau du tournoi ?

In [6]:
print(
    rg.groupby('Round')
        .agg( # NamedAgg permet de nommer les agrégations
            nb_ace_moy=pd.NamedAgg(column="Ace_tot", aggfunc="mean")
        )
)

       nb_ace_moy
Round            
1       13.476190
2       13.193548
3       12.562500
4        9.125000
5        7.000000
6       10.000000
7        6.000000


5. Filtrer les matchs pour lesquels au moins une des variables `DBF.1` et `DBF.2` est manquante.

In [7]:
rg[rg['DBF.1'].isna()|rg['DBF.2'].isna()].filter(items=["Player1", "Player2",'DBF.1','DBF.2'])

,Player1,Player2,DBF.1,DBF.2
56,Simone Bolelli,Yen-Hsun Lu,NaN,NaN
63,Somdev Devvarman,Roger Federer,NaN,NaN


6. Remplacer les valeurs manquantes de `DBF.1` par zéro avec la méthode `loc`.

In [8]:
rg.loc[rg['DBF.1'].isna(), "DBF.1"] = 0
print(rg[rg['DBF.1'].isna()|rg['DBF.2'].isna()].filter(items=["Player1", "Player2",'DBF.1','DBF.2']))

             Player1        Player2  DBF.1  DBF.2
56    Simone Bolelli    Yen-Hsun Lu    0.0    NaN
63  Somdev Devvarman  Roger Federer    0.0    NaN


7. Remplacer les valeurs manquantes de `DBF.2` par zéro avec la méthode `fillna`.

In [9]:
rg['DBF.2'].fillna(0, inplace=True)

/tmp/ipykernel_3968/2457930233.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  rg['DBF.2'].fillna(0, inplace=True)


8. Extraire la liste des participants à partir des colonnes `Player1` et `Player2`. Une façon de faire consiste à utiliser `concat` et la méthode `drop_duplicates` pour obtenir le résultat sous la forme d'une série et de la convertir en dataframe avec la méthode `to_frame`.

In [30]:
rg_joueurs=pd.concat([rg['Player1'],rg['Player2']]).drop_duplicates().to_frame(name="Joueurs").reset_index(drop=True)
print(rg_joueurs)

                 Joueurs
0    Pablo Carreno-Busta
1       Somdev Devvarman
2           Tobias Kamke
3       Julien Benneteau
4            Lukas Lacko
..                   ...
122          Yen-Hsun Lu
123      Grigor Dimitrov
124          Guido Pella
125         David Goffin
126     Janko Tipsarevic

[127 rows x 1 columns]


9. Écrire une fonction `n_match` qui prend une chaîne de caractères `joueur` en entrée et retourne le nombre de matchs disputés par le joueur.

In [28]:
def n_match(s):
    s=s.upper()
    rg_temp=rg.filter(items=["Player1", "Player2"]).copy()
    rg_temp['Player1']=rg_temp['Player1'].str.upper()
    rg_temp['Player2']=rg_temp['Player2'].str.upper()
    return len(rg_temp[(rg_temp['Player1']==s)|(rg_temp['Player2']==s)])

print(n_match('Rafael Nadal'))

7


10. Utiliser les deux question précédentes et la méthode `apply` pour compter le nombre de matchs que chaque participant a disputé et ordonner le résultat par ordre décroissant.

In [32]:
rg_joueurs['n_match']=rg_joueurs['Joueurs'].apply(n_match)
print(rg_joueurs.sort_values(by='n_match',ascending=False))

                    Joueurs  n_match
47             Rafael Nadal        7
80             David Ferrer        7
62           Novak Djokovic        6
15       Jo-Wilfried Tsonga        6
54               Tommy Haas        5
..                      ...      ...
114           Daniel Brands        1
122             Yen-Hsun Lu        1
120         Guillaume Rufin        1
119  Guillermo Garcia-Lopez        1
125            David Goffin        1

[127 rows x 2 columns]


11. Charger maintenant le jeu de données relatif au tournoi de Wimbledon dans un dataframe `wb` à partir du fichier `wimbledon2013.csv`.

In [33]:
wb=pd.read_csv('data/wimbledon2013.csv',sep=',')
wb.head(10)

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,BPC.2,BPW.2,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2
0,B.Becker,A.Murray,1,0,0,3,59,29,41,14,...,10,5,23,17,NaN,6,6,6,NaN,NaN
1,J.Ward,Y-H.Lu,1,0,1,3,62,77,38,35,...,15,2,46,39,NaN,6,6,7,7.0,NaN
2,N.Mahut,J.Hajek,1,1,3,0,72,44,28,10,...,1,0,19,12,NaN,2,4,3,NaN,NaN
3,T.Robredo,A.Bogomolov Jr.,1,1,3,0,77,40,23,12,...,0,0,22,13,NaN,2,2,4,NaN,NaN
4,R.Haase,M.Youzhny,1,0,0,3,68,61,32,15,...,21,3,44,30,NaN,6,7,7,NaN,NaN
5,M.Gicquel,V.Pospisil,1,0,0,3,59,41,41,27,...,12,4,33,26,NaN,6,6,7,NaN,NaN
6,A.Kuznetsov,A.Montanes,1,1,3,1,63,56,37,21,...,9,2,11,10,NaN,3,4,6,3.0,NaN
7,J.Tipsarevic,V.Troicki,1,0,0,3,61,47,39,21,...,10,2,38,27,NaN,6,6,7,NaN,NaN
8,M.Baghdatis,M.Cilic,1,0,0,3,61,31,39,16,...,6,4,11,8,NaN,6,6,6,NaN,NaN
9,K.De Schepper,P.Lorenzi,1,1,3,0,67,56,33,21,...,6,0,23,15,NaN,6,4,2,NaN,NaN


12. Ajouter une colonne `Tournoi` dans les dataframes `rg` et `wb` contenant respectivement les chaînes de caractères `"RG"` et `"WB"`.

In [34]:
wb['Tournoi']='WB'
rg['Tournoi']='RG'

13. Concaténer les deux dataframes dans un nouveau dataframe `tennis`.

In [35]:
tennis=pd.concat([rg,wb],ignore_index=True)
tennis.head(10)

,Player1,Player2,Round,Result,FNL.1,FNL.2,FSP.1,FSW.1,SSP.1,SSW.1,...,NPA.2,NPW.2,TPW.2,ST1.2,ST2.2,ST3.2,ST4.2,ST5.2,Ace_tot,Tournoi
0,Pablo Carreno-Busta,Roger Federer,1,0,0,3,62,27,38,11,...,14,18,88.0,6,6,6.0,NaN,NaN,11.0,RG
1,Somdev Devvarman,Daniel Munoz-De La Nava,1,1,3,0,62,54,38,22,...,22,25,106.0,3,3,5.0,NaN,NaN,7.0,RG
2,Tobias Kamke,Paolo Lorenzi,1,1,3,2,62,53,38,15,...,19,27,139.0,3,3,6.0,6.0,3.0,10.0,RG
3,Julien Benneteau,Ricardas Berankis,1,1,3,1,72,87,28,19,...,33,43,149.0,6,3,7.0,6.0,NaN,27.0,RG
4,Lukas Lacko,Sam Querrey,1,0,0,3,52,31,48,22,...,12,13,93.0,6,6,6.0,NaN,NaN,14.0,RG
5,Jan Hajek,Denis Kudla,1,1,3,1,70,58,30,18,...,6,9,93.0,2,7,0.0,4.0,NaN,9.0,RG
6,Adrian Mannarino,Pablo Cuevas,1,0,2,3,63,71,37,38,...,14,22,175.0,6,2,6.0,5.0,7.0,10.0,RG
7,Gilles Simon,Lleyton Hewitt,1,1,3,2,59,42,41,25,...,19,35,120.0,6,6,4.0,1.0,5.0,11.0,RG
8,Philipp Petzschner,Marin Cilic,1,0,0,3,56,27,44,13,...,13,21,92.0,6,6,6.0,NaN,NaN,5.0,RG
9,Radek Stepanek,Nick Kyrgios,1,0,0,3,63,62,37,29,...,9,20,130.0,7,7,7.0,NaN,NaN,15.0,RG


14. Utiliser le dataframe `tennis` pour comparer le nombre moyen d'aces par match à chaque niveau du tournoi à Roland Garros et à Wimbledon. Afficher le résultat en format large.

In [ ]:
tennis['Ace_tot']=tennis['ACE.1']+tennis['ACE.2']

#format long
print(
    tennis.groupby(['Tournoi','Round'])
        .agg( # NamedAgg permet de nommer les agrégations
            nb_ace_moy=pd.NamedAgg(column="Ace_tot", aggfunc="mean")
        )
        .reset_index()
)

#format large
print(
    tennis.groupby(['Tournoi','Round'])
        .agg( # NamedAgg permet de nommer les agrégations
            nb_ace_moy=pd.NamedAgg(column="Ace_tot", aggfunc="mean")
        )
        .reset_index()
        .pivot(
            index="Tournoi",
            columns="UniqueCarrier",
            values="count"
        )

)


   Tournoi  Round  nb_ace_moy
0       RG      1   13.476190
1       RG      2   13.193548
2       RG      3   12.562500
3       RG      4    9.125000
4       RG      5    7.000000
5       RG      6   10.000000
6       RG      7    6.000000
7       WB      1   21.125000
8       WB      2   23.869565
9       WB      3   24.000000
10      WB      4   24.375000
11      WB      5   26.500000
12      WB      6   27.500000
13      WB      7   13.000000


15. Quelle différence y a-t-il dans le format des noms des joueurs entre les dataframes `rg` et `wb` ?

16. Construire un dataframe `rg_victoires` avec les trois colonnes suivantes pour le tournoi de Roland Garros :
- `joueur` : nom du joueur tel qu'il est donné dans `rg`,
- `nom_joueur` : nom de famille du joueur uniquement,
- `n_victoire` : nombre de matchs gagnés dans le tournoi.

In [43]:
joueur= "Rafael Nadal"


def n_victoire(df,j):
    n_victoire1=len(df[(df["Player1"]==joueur)&(df["Result"]==1)])
    n_victoire2=len(df[(df["Player2"]==joueur)&(df["Result"]==0)])
    return n_victoire1+n_victoire2




In [46]:
rg_victoires=pd.concat([rg['Player1'],rg['Player2']]).drop_duplicates().to_frame(name="Joueurs").reset_index(drop=True)
rg_victoires['nom_joueur']=rg_victoires['Joueurs'].str.split(' ').apply(lambda v:v[-1])
rg_victoires['n_victoire']=rg_victoires['Joueurs'].apply(lambda joueur: n_victoire(rg,joueur))

rg_victoires.head(5)

,Joueurs,nom_joueur,n_victoire
0,Pablo Carreno-Busta,Carreno-Busta,7
1,Somdev Devvarman,Devvarman,7
2,Tobias Kamke,Kamke,7
3,Julien Benneteau,Benneteau,7
4,Lukas Lacko,Lacko,7


17. Construire un dataframe `wb_victoires` avec les trois colonnes suivantes pour le tournoi de Wimbledon :
- `joueur` : nom du joueur tel qu'il est donné dans `wb`,
- `nom_joueur` : nom de famille du joueur uniquement,
- `n_victoire` : nombre de matchs gagnés dans le tournoi.

In [47]:
wb_victoires=pd.concat([wb['Player1'],wb['Player2']]).drop_duplicates().to_frame(name="Joueurs").reset_index(drop=True)
wb_victoires['nom_joueur']=wb_victoires['Joueurs'].str.split('.').apply(lambda v:v[-1])
wb_victoires['n_victoire']=wb_victoires['Joueurs'].apply(lambda joueur: n_victoire(wb,joueur))

wb_victoires.head(5)

,Joueurs,nom_joueur,n_victoire
0,B.Becker,Becker,0
1,J.Ward,Ward,0
2,N.Mahut,Mahut,0
3,T.Robredo,Robredo,0
4,R.Haase,Haase,0


18. Faire une jointure entre `rg_victoires` et `wb_victoires` sur la colonne `nom_joueur` pour comparer le nombre de victoires par tournoi pour chaque joueur. Expliquer la différence de résultat selon que la jointure est à gauche, à droite, intérieure ou extérieure.

In [51]:
tennis_victoires=rg_victoires.merge(wb_victoires,suffixes=('_rg','_wb'),on='nom_joueur',how='left')
tennis_victoires.head(5)

,Joueurs_rg,nom_joueur,n_victoire_rg,Joueurs_wb,n_victoire_wb
0,Pablo Carreno-Busta,Carreno-Busta,7,NaN,NaN
1,Somdev Devvarman,Devvarman,7,NaN,NaN
2,Tobias Kamke,Kamke,7,T.Kamke,0.0
3,Julien Benneteau,Benneteau,7,J.Benneteau,0.0
4,Lukas Lacko,Lacko,7,L.Lacko,0.0
